# HarborSense AI: Exploratory Data Analysis (EDA)
## Port Congestion & Delay Dynamics Analysis

**Objective:** Inspect data distributions, verify schema integrity, validate the 6.0-hour congestion threshold, check for target leakage, and evaluate port-to-port variability.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from harborsense.config import config
from harborsense.data_generation import generate_dataset
from harborsense.eda import run_eda

### 1. Dataset Generation and Schema Inspection

In [ ]:
if config.data_file.exists():
    df = pd.read_csv(config.data_file)
else:
    df = generate_dataset(n_steps_per_port=300, seed=config.random_seed, output_path=config.data_file)

print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Unique Simulated Ports: {df['port_id'].unique().tolist()}")
df.head()

### 2. Summary Statistics & Data Quality Checks
Verify missing values, duplicates, and physical domain bounds.

In [ ]:
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")

print("\nSummary Statistics:")
display(df.describe().T)

### 3. Congestion Class Balance & Target Validation
Verify target rule: `congestion = 1 if future_median_wait_hours > 6.0 else 0`.

In [ ]:
threshold_check = (df["congestion"] == (df["future_median_wait_hours"] > 6.0).astype(int)).all()
print(f"Congestion threshold (6.0h) perfectly consistent: {threshold_check}")

print("\nClass Counts and Proportions:")
print(df["congestion"].value_counts())
print(df["congestion"].value_counts(normalize=True))

### 4. Congestion Rate Across Simulated Ports
Analyze port-to-port differences due to differing berth scales and arrival demands.

In [ ]:
port_stats = df.groupby("port_id").agg(
    total_observations=("congestion", "count"),
    congestion_rate=("congestion", "mean"),
    mean_wait_hours=("future_median_wait_hours", "mean"),
    median_wait_hours=("future_median_wait_hours", "median"),
    mean_arrivals=("expected_arrivals_24h", "mean"),
    berth_capacity=("berth_capacity", "first"),
).reset_index()
display(port_stats)

### 5. Waiting Time Distribution & Threshold Analysis

In [ ]:
plt.figure(figsize=(8, 4.5), dpi=120)
plt.hist(df["future_median_wait_hours"], bins=35, color="#3182ce", edgecolor="white", alpha=0.8)
plt.axvline(6.0, color="red", linestyle="--", lw=2, label="Congestion Threshold (6.0h)")
plt.xlabel("Future Median Waiting Time (Hours)")
plt.ylabel("Frequency")
plt.title("Distribution of Realized Future Waiting Times")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6. Temporal & Circadian Patterns

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
df.groupby("hour")["congestion"].mean().plot(ax=ax1, marker="o", color="#2b6cb0", lw=2)
ax1.set_title("Congestion Rate by Hour of Day")
ax1.set_xlabel("Hour (0-23)")
ax1.set_ylabel("Probability")
ax1.grid(True, alpha=0.3)

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
day_rates = df.groupby("day_of_week")["congestion"].mean()
ax2.bar([days[i] for i in day_rates.index], day_rates.values, color="#4299e1", edgecolor="#2b6cb0")
ax2.set_title("Congestion Rate by Day of Week")
ax2.set_ylabel("Probability")
ax2.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 7. Feature Correlation & Target Relationships

In [ ]:
num_cols = [
    "expected_arrivals_24h",
    "berth_capacity",
    "current_utilization",
    "current_queue_length",
    "recent_arrivals_6h",
    "weather_severity",
    "historical_delay_rate",
    "future_median_wait_hours",
    "congestion",
]
corr = df[num_cols].corr()

plt.figure(figsize=(8, 7), dpi=120)
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(num_cols)), [c.replace('_', ' ').title() for c in num_cols], rotation=45, ha="right")
plt.yticks(range(len(num_cols)), [c.replace('_', ' ').title() for c in num_cols])
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### 8. Key EDA Findings & Feature Engineering Takeaways
1. **Nonlinear Queue Sensitivity:** When `current_queue_length` and `expected_arrivals_24h` exceed nominal capacity, waiting times increase rapidly.
2. **Weather Capacity Derate:** Severe weather interacts multiplicatively with high utilization, justifying an explicit `weather_utilization_interaction` feature.
3. **Zero Target Leakage:** Historical delay rates are computed from past observations only; no future outcomes bleed into inference matrices.